# 04 — AI autonomy quality

**Purpose.** Score Aura's autonomous and semi-autonomous actions for relevance and quality.
**KPI validated.** AI autonomy quality target ≥85% (`plan.md` section 22).

Method: 100 actions sampled at random from `pilot/coded/autonomy/actions_sample_100.csv`. Three raters (Shaurya, Shorya, one external) rate each on a 1-5 Likert plus a binary `correct` flag, blind to each other. We report:
- Mean Likert and percent-correct per rater.
- Inter-rater agreement via Cohen's kappa (pairwise) and Fleiss' kappa (3-way) on the binary flag.
- Aggregate quality score (majority vote on `correct`).


In [ ]:
import json
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200
ACCENT = "#FF5B2E"

SEED = 20260507
rng = np.random.default_rng(SEED)
CHART_DIR = Path("../charts"); CHART_DIR.mkdir(exist_ok=True)
CODED = Path("../../coded/autonomy")


## Data load

Real path: `pilot/coded/autonomy/ratings_rater{1,2,3}.csv`. Each row: `action_id`, `likert_1to5`, `correct_binary`.


## SYNTHETIC DATA — REPLACE WITH REAL
Three correlated rater streams over 100 actions, with ~88% true-correct rate and rater agreement around kappa = 0.7.


In [ ]:
N_ACTIONS = 100
action_ids = [f"A{i:03d}" for i in range(1, N_ACTIONS + 1)]

def synth_ratings(n=N_ACTIONS, true_correct_rate=0.88, rater_noise=0.10, seed=SEED):
    rs = np.random.default_rng(seed)
    true_correct = rs.binomial(1, true_correct_rate, size=n)
    true_likert = np.where(true_correct == 1, rs.normal(4.4, 0.5, n), rs.normal(2.6, 0.7, n))
    true_likert = np.clip(np.round(true_likert), 1, 5).astype(int)
    raters = {}
    for r in (1, 2, 3):
        noise_flip = rs.binomial(1, rater_noise, size=n)
        rater_correct = np.where(noise_flip == 1, 1 - true_correct, true_correct)
        rater_likert = np.clip(true_likert + rs.integers(-1, 2, size=n), 1, 5)
        raters[r] = pd.DataFrame({
            "action_id": action_ids,
            f"likert_r{r}": rater_likert,
            f"correct_r{r}": rater_correct,
        })
    return raters

if all((CODED / f"ratings_rater{r}.csv").exists() for r in (1, 2, 3)):
    r1 = pd.read_csv(CODED / "ratings_rater1.csv")
    r2 = pd.read_csv(CODED / "ratings_rater2.csv")
    r3 = pd.read_csv(CODED / "ratings_rater3.csv")
else:
    ratings = synth_ratings()
    r1, r2, r3 = ratings[1], ratings[2], ratings[3]

merged = r1.merge(r2, on="action_id").merge(r3, on="action_id")
print("rows:", len(merged))
merged.head()


## Per-rater summary


In [ ]:
for r in (1, 2, 3):
    mean_l = merged[f"likert_r{r}"].mean()
    pct_c = merged[f"correct_r{r}"].mean() * 100
    print(f"rater {r}: mean Likert = {mean_l:.2f},  percent correct = {pct_c:.1f}%")


## Pairwise Cohen's kappa on the binary `correct` flag


In [ ]:
pair_kappas = []
for a, b in combinations((1, 2, 3), 2):
    k = cohen_kappa_score(merged[f"correct_r{a}"], merged[f"correct_r{b}"])
    pair_kappas.append({"pair": f"r{a} vs r{b}", "cohens_kappa": round(float(k), 3)})
print(pd.DataFrame(pair_kappas).to_string(index=False))


## Fleiss' kappa (3-rater) on the binary `correct` flag


In [ ]:
# statsmodels expects a (n_subjects x n_categories) count table
cat_table = np.zeros((len(merged), 2), dtype=int)
for r in (1, 2, 3):
    for i, v in enumerate(merged[f"correct_r{r}"].values):
        cat_table[i, int(v)] += 1
fk = float(fleiss_kappa(cat_table))
print(f"Fleiss' kappa (3 raters, binary correct) = {fk:.3f}")


## Aggregate quality score

Majority vote across the three raters on `correct`. The KPI target is >=85% majority-correct.


In [ ]:
merged["majority_correct"] = (
    merged[["correct_r1", "correct_r2", "correct_r3"]].sum(axis=1) >= 2
).astype(int)
majority_pct = merged["majority_correct"].mean() * 100
print(f"majority-correct = {majority_pct:.1f}% (n={len(merged)} actions)")
print(f"meets >=85% target: {majority_pct >= 85}")


## Results — interpretation

- Fleiss' kappa above 0.6 is "substantial agreement" by Landis-Koch; above 0.8 is "almost perfect".
- The aggregate majority-correct percentage is the headline number for slide 9.
- If majority-correct lands below 85%, error analysis on the disagreed-on rows tells us where the agents misjudged urgency or context.


## Charts

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))
means = [merged[f"likert_r{r}"].mean() for r in (1, 2, 3)]
ax.bar([f"rater {r}" for r in (1, 2, 3)], means, color=ACCENT, alpha=0.9)
ax.set_ylim(0, 5)
ax.set_ylabel("Mean Likert (1-5)")
ax.set_title("Per-rater mean quality rating across 100 actions")
fig.tight_layout()
fig.savefig(CHART_DIR / "04_rater_means.png")
plt.show()

fig, ax = plt.subplots(figsize=(6.4, 4.0))
labels = [d["pair"] for d in pair_kappas] + ["Fleiss kappa (3-way)"]
values = [d["cohens_kappa"] for d in pair_kappas] + [fk]
ax.bar(labels, values, color=ACCENT, alpha=0.9)
ax.axhline(0.6, color="#0E0E0E", linewidth=0.8, linestyle="--", label="substantial (0.6)")
ax.axhline(0.8, color="#0E0E0E", linewidth=0.8, linestyle=":", label="almost perfect (0.8)")
ax.set_ylabel("Kappa")
ax.set_title("Inter-rater agreement on action correctness")
ax.legend()
fig.tight_layout()
fig.savefig(CHART_DIR / "04_kappa.png")
plt.show()
